Setup

---



In [ ]:
!python --version
!java -version

In [ ]:
%pip -q install pyspark

In [ ]:
#Import Spark and create a SparkSession

from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.appName("Our First Spark Example").master("local[*]").getOrCreate()

spark

In [ ]:
# Upload the existing CSV file

from google.colab import files

uploaded = files.upload()

CSV_PATH = next(iter(uploaded))

print("Uploaded file:", CSV_PATH)

In [ ]:
#Read the CSV into a Spark DataFrame

df = spark.read.csv(CSV_PATH, header=True, inferSchema=True)
print("CSV file loaded successfully.")

In [ ]:
#Check the number of rows and columns

print("Number of rows:", df.count())
print("Number of columns:", len(df.columns))

In [ ]:
#Display the first five rows

df.show(5, truncate=False)

In [ ]:
#View the DataFrame schema

df.printSchema()

In [ ]:
#Convert the transaction date when inferShema=False

df.select(F.to_date(df.transaction_date).alias("transaction_date")).show(5)

In [ ]:
#Display summary statistics

df.select("customer_age", "quantity", "unit_price", "total_amount").describe().show()

PySpark Operations

---



In [ ]:
# PySpark Operation 1 – Select columns

selected_data = df.select(
    "transaction_id",
    "city",
    "product_category",
    "total_amount"
)

selected_data.show(10, truncate=False)

In [ ]:
# PySpark Operation 2 – Filter completed orders

completed_orders = df.filter(df.order_status == "Completed")

print("Number of completed orders:", completed_orders.count())
completed_orders.show(10, truncate=False)

In [ ]:
# PySpark Operation 3 – Group by product category and sum revenue

category_revenue = completed_orders.groupBy("product_category").sum("total_amount")

category_revenue.orderBy(
    F.desc("sum(total_amount)")
).show(truncate=False)


Spark SQL Operations

---



In [ ]:
# Create a temporary view of the DataFrame
df.createOrReplaceTempView("sales_data")

In [ ]:
# Spark SQL Operation 1: SELECT & WHERE
# Retrieves specific columns and filters for completed orders
# ---------------------------------------------------------
sql_op1 = spark.sql("""
    SELECT transaction_id, product_category, total_amount, order_status
    FROM sales_data
    WHERE order_status = 'Completed'
""")
print("Spark SQL Op 1: Completed Orders")
sql_op1.show(5)

In [ ]:
# Spark SQL Operation 2: GROUP BY & SUM()
# Calculates the total sales revenue for each product category
# ---------------------------------------------------------
sql_op2 = spark.sql("""
    SELECT product_category, SUM(total_amount) AS total_revenue
    FROM sales_data
    WHERE order_status = 'Completed'
    GROUP BY product_category
""")
print("Spark SQL Op 2: Total Revenue by Category")
sql_op2.show()

In [ ]:
# Spark SQL Operation 3: ORDER BY
# Arranges the grouped results in descending order
# to easily spot the top-performing categories
# ---------------------------------------------------------
sql_op3 = spark.sql("""
    SELECT product_category, SUM(total_amount) AS total_revenue
    FROM sales_data
    WHERE order_status = 'Completed'
    GROUP BY product_category
    ORDER BY total_revenue DESC
""")
print("Spark SQL Op 3: Highest Revenue Categories Sorted")
sql_op3.show()